# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM014 - Distance travelled to work by age |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |RM014_Distance_travelled_to_work_by_age |
| Output File |travel_distance_by_age.json |
| Notebook Author | Peter |
| Last Updated |2026-07-15 |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [4]:
# File paths

input_file = "RM014_Distance_travelled_to_work_by_age.xlsx"
output_file = "travel_distance_by_age.json"

raw_data_path = Path("../../data/demographic/raw/travel distance")
output_data_path = Path("../../data/demographic/processed/travel distance")

## 3. Load Dataset

In [5]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Distance travelled to work (8 categories) Code,Distance travelled to work (8 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0,Aged under 1 year,918
1,E06000001,Hartlepool,-8,Does not apply,1,Aged 1 year,954
2,E06000001,Hartlepool,-8,Does not apply,2,Aged 2 years,1046
3,E06000001,Hartlepool,-8,Does not apply,3,Aged 3 years,988
4,E06000001,Hartlepool,-8,Does not apply,4,Aged 4 years,1077


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [6]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220848 entries, 0 to 220847
Data columns (total 7 columns):
 #   Column                                          Non-Null Count   Dtype 
---  ------                                          --------------   ----- 
 0   Lower tier local authorities Code               220848 non-null  object
 1   Lower tier local authorities                    220848 non-null  object
 2   Distance travelled to work (8 categories) Code  220848 non-null  int64 
 3   Distance travelled to work (8 categories)       220848 non-null  object
 4   Age (86 categories) Code                        220848 non-null  int64 
 5   Age (86 categories)                             220848 non-null  object
 6   Observation                                     220848 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 11.8+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Distance travelled to work (8 categories) Code',
       'Distance travelled to work (8 categories)', 'Age (86 categories) Code',
       'Age (86 categories)', 'Observation'],
      dtype='object')

In [7]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Distance travelled to work (8 categories) Code,Distance travelled to work (8 categories),Age (86 categories) Code,Age (86 categories),Observation
count,220848,220848,220848.000000,220848,220848.000000,220848,220848.000000
unique,321,321,NaN,8,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Aged under 1 year,NaN
freq,688,688,NaN,27606,NaN,2568,NaN
mean,NaN,NaN,2.500000,NaN,42.500000,NaN,266.572575
std,NaN,NaN,4.387492,NaN,24.824439,NaN,594.604736
min,NaN,NaN,-8.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,1.750000,NaN,21.000000,NaN,4.000000
50%,NaN,NaN,3.500000,NaN,42.500000,NaN,63.000000
75%,NaN,NaN,5.250000,NaN,64.000000,NaN,273.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [8]:
# Cleaning code

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [9]:
selected_columns = [
    'Lower tier local authorities',
    'Distance travelled to work (8 categories)',
    'Age (86 categories) Code',
    'Observation',
]

df_processed = df_processed[selected_columns]
df_processed = df_processed.rename(columns={
    'Distance travelled to work (8 categories)': 'Distance travelled to work',
    'Age (86 categories) Code': 'Age'})

In [10]:
# Dataset-specific processing
from ontology.age_ontology import (
    age_group,
    age_order
)

In [11]:
def get_travel_distance_by_age(local_authority):
    region_data = df_processed[
        df_processed["Lower tier local authorities"] == local_authority
    ].copy()


    # Create your dissertation age groups
    region_data["Age_Group"] = (
        region_data["Age"]
        .apply(age_group)
    )

    # Group by age group and commuting distance
    grouped_distance_age = (
        region_data
        .groupby([
            "Age_Group",
            "Distance travelled to work"
        ])["Observation"]
        .sum()
        .reset_index()
    )

    # Sort age groups properly
    grouped_distance_age["Age_Group"] = pd.Categorical(
        grouped_distance_age["Age_Group"],
        categories=age_order,
        ordered=True
    )

    return grouped_distance_age

In [12]:
get_travel_distance_by_age("Manchester")

,Age_Group,Distance travelled to work,Observation
0,0-4,10km to less than 30km,0
1,0-4,30km to less than 60km,0
2,0-4,5km to less than 10km,0
3,0-4,60km and over,0
4,0-4,Does not apply,34378
5,0-4,Less than 5km,0
6,0-4,"Works mainly at an offshore installation, in n...",0
7,0-4,Works mainly from home,0
8,12-17,10km to less than 30km,58
9,12-17,30km to less than 60km,8


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [13]:
get_travel_distance_by_age("Manchester")['Observation'].sum()

np.int64(551947)

In [14]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220848 entries, 0 to 220847
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Lower tier local authorities  220848 non-null  object
 1   Distance travelled to work    220848 non-null  object
 2   Age                           220848 non-null  int64 
 3   Observation                   220848 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 6.7+ MB


Lower tier local authorities    0
Distance travelled to work      0
Age                             0
Observation                     0
dtype: int64

## 8. Export Processed Dataset

In [15]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

travel_distance_by_age = get_travel_distance_by_age("Manchester")
travel_distance_by_age.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported